# 02 Ingest — Azure Storage (parquet · full & incremental)

**Doel:** Lees alle actieve `azurestorage`-rijen uit de control table en laad de
bijbehorende parquet-bestanden als Delta-tabellen in `STAGING_AZURESTORAGE`.

**Laadstrategieën (gestuurd door `load_type` in de control table):**

| `load_type` | Gedrag |
|---|---|
| `full` | Alle bestanden opnieuw inlezen en de doeltabel volledig overschrijven |
| `incremental` | Auto Loader (`cloudFiles`) — alleen nieuwe bestanden verwerken, checkpoint per doeltabel |

**Switchen tussen full en incrementeel:** Eén `UPDATE` in de control table is genoeg —
geen codewijziging nodig.

```sql
UPDATE DEMO_DEV.CONFIG.pipeline_sources
SET    load_type = 'incremental'
WHERE  target_table = 'order_header'
```

**Checkpoints** (incremental mode): `{source_path}/_checkpoints/{target_table}/`

**Audit-kolommen per doeltabel:**

| Kolom | Bron |
|---|---|
| `_ingestion_timestamp` | `current_timestamp()` |
| `_source_system` | waarde uit control table |
| `_source_file` | `_metadata.file_path` |
| `_last_modified` | `_metadata.file_modification_time` |
| `_pipeline_run_id` | Workflow job run id (widget) |

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO_DEV", "Catalog")
dbutils.widgets.text("pipeline_run_id", "", "Pipeline Run ID (ingevuld door Workflow)")

catalog = dbutils.widgets.get("catalog")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

print(f"Catalog          : {catalog}")
print(f"Pipeline run id  : {pipeline_run_id or '(niet opgegeven — handmatig run)'}")

## Stap 1 — Control table inlezen

In [ ]:
control_table = f"{catalog}.CONFIG.pipeline_sources"

sources_df = spark.sql(
    f"""
    SELECT *
    FROM   {control_table}
    WHERE  source_system = 'azurestorage'
    AND    is_active     = true
    """
)

source_rows = sources_df.collect()
print(f"Actieve azurestorage-bronnen gevonden: {len(source_rows)}")
for row in source_rows:
    print(f"  → {row['target_table']} ({row['load_type']}) — {row['file_pattern']}")

## Stap 2 — Ingest per bron

In [ ]:
from pyspark.sql import functions as F

row_count_report = []

for row in source_rows:
    source_path   = row["source_path"]
    file_pattern  = row["file_pattern"]
    target_schema = row["target_schema"]
    target_table  = row["target_table"]
    load_type     = row["load_type"]
    source_system = row["source_system"]

    full_target = f"{catalog}.{target_schema}.{target_table}"
    print(f"\n--- Verwerken: {full_target} (load_type={load_type}) ---")

    if load_type == "full":
        # -----------------------------------------------------------------
        # FULL LOAD — lees alle bestanden die overeenkomen met het glob-filter
        # en overschrijf de doeltabel volledig.
        # includeMetadata=True geeft toegang tot _metadata.file_path en
        # _metadata.file_modification_time voor de audit-kolommen.
        # -----------------------------------------------------------------
        raw_df = (
            spark.read.format("parquet")
            .option("pathGlobFilter", file_pattern)
            .option("recursiveFileLookup", "false")
            .load(source_path)
        )

        # Voeg de vijf audit-kolommen toe (CONTEXT.md §5).
        enriched_df = (
            raw_df
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_source_system",       F.lit(source_system))
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
            .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
            # _metadata is een struct-kolom die niet naar Delta mag worden geschreven.
            .drop("_metadata")
        )

        # Schrijf als Delta (full = overschrijven).
        (
            enriched_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_target)
        )

        written_count = spark.table(full_target).count()
        row_count_report.append((full_target, load_type, written_count))
        print(f"  [full] Geschreven: {written_count:,} rijen → {full_target}")

    elif load_type == "incremental":
        # -----------------------------------------------------------------
        # INCREMENTAL LOAD — Auto Loader (cloudFiles) met checkpoint.
        # Checkpoints leven onder _checkpoints/{target_table}/ binnen het
        # external volume.  Herdraaien pikt alleen nieuwe bestanden op;
        # er worden geen dubbele rijen geschreven.
        # -----------------------------------------------------------------
        checkpoint_path = f"{source_path}/_checkpoints/{target_table}"

        print(f"  Checkpoint pad   : {checkpoint_path}")
        print(f"  Bestandsfilter   : {file_pattern}")

        # Rijentelling vóór de run — voor delta row-count in de output.
        rows_before = (
            spark.table(full_target).count()
            if spark.catalog.tableExists(full_target)
            else 0
        )

        # Auto Loader leest incrementeel via cloudFiles.
        # trigger(availableNow=True) verwerkt alle beschikbare (nieuwe) bestanden
        # en stopt dan automatisch — geschikt voor batch Workflows.
        stream_query = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format",         "parquet")
            .option("cloudFiles.schemaLocation", checkpoint_path)
            .option("pathGlobFilter",            file_pattern)
            .option("recursiveFileLookup",       "false")
            .load(source_path)
            # Voeg audit-kolommen toe (CONTEXT.md §5).
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_source_system",       F.lit(source_system))
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
            .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
            .drop("_metadata")
            .writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema",         "true")
            .trigger(availableNow=True)
            .toTable(full_target)
        )

        # Wacht tot alle beschikbare bestanden verwerkt zijn.
        stream_query.awaitTermination()

        rows_after  = spark.table(full_target).count()
        delta_count = rows_after - rows_before
        row_count_report.append((full_target, load_type, delta_count))
        print(f"  [incremental] Nieuwe rijen: {delta_count:,} | Totaal: {rows_after:,} → {full_target}")
        print(f"  Checkpoint opgeslagen onder: {checkpoint_path}")

    else:
        raise ValueError(
            f"Onbekend load_type='{load_type}' voor tabel '{target_table}'. "
            "Verwacht 'full' of 'incremental'."
        )

## Stap 3 — Validatie & row counts

In [ ]:
print("\n=== Row count samenvatting ===")
for table, mode, count in row_count_report:
    label = "geschreven" if mode == "full" else "nieuwe rijen"
    print(f"  [{mode}] {table}: {count:,} {label}")

## Resultaat

Alle actieve `azurestorage`-bronnen zijn ingeladen als Delta-tabellen in
`STAGING_AZURESTORAGE`. Elke tabel bevat de vijf audit-kolommen:
`_ingestion_timestamp`, `_source_system`, `_source_file`, `_last_modified`,
`_pipeline_run_id`.

**Full mode:** Alle bestanden opnieuw verwerkt, doeltabel volledig overschreven.

**Incremental mode:** Alleen nieuwe bestanden verwerkt via Auto Loader (`cloudFiles`).
Checkpoints staan onder `_checkpoints/{target_table}/` in het external volume.
Herdraaien levert uitsluitend de delta — geen dubbele rijen.

**Demo-tip:** Verander `load_type` in de control table met één `UPDATE` en draai het
Workflow opnieuw — de notebook past zijn gedrag automatisch aan zonder codewijziging.